# Import si incarcare date

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("output/fact_agriculture_analytic.xlsx")
print("Dimensiune:", df.shape)
display(df.head())

Dimensiune: (48514, 17)


,FactID,time,temperature,humidity,pressure,pressure_vapor,radiation,precipitation,wind_speed,vane_direction,soil_time,temperature_depth_0,temperature_depth_1,temperature_depth_5,moisture_w-o_salinity_depth_5,time_difference_min,has_soil_data
0,1,2025-07-14 12:51:47,26.5,59.1,98.34,2.046,382.551,0.0,6.4,270,2025-07-14 12:50:31,30.06,30.07,21.74,33.74,1.266667,True
1,2,2025-07-14 12:56:46,26.8,57.1,98.33,2.009,366.046,0.0,3.2,225,2025-07-14 12:55:31,30.18,30.18,21.76,33.74,1.250000,True
2,3,2025-07-14 13:01:45,26.9,56.4,98.33,1.992,389.825,0.0,4.0,270,2025-07-14 13:00:28,30.17,30.14,21.76,33.73,1.283333,True
3,4,2025-07-14 13:06:45,26.8,56.1,98.33,1.975,377.796,0.0,11.2,270,2025-07-14 13:05:28,30.17,30.13,21.78,33.70,1.283333,True
4,5,2025-07-14 13:11:45,26.7,56.9,98.32,1.992,441.577,0.0,12.0,270,2025-07-14 13:10:31,29.99,29.99,21.77,33.67,1.233333,True


# Selectarea variabilelor

In [3]:
target = "temperature_depth_5"
features = [
    "temperature",
    "humidity",
    "pressure",
    "pressure_vapor",
    "radiation",
    "precipitation",
    "wind_speed",
    "vane_direction"
]
model_data = (
    df[df["has_soil_data"] == True]
    .sort_values(["soil_time", "time"])
    .drop_duplicates(subset=["soil_time"], keep="first")
    .dropna(subset=features + [target])
    .copy()
)
print("Randuri disponibile:", len(model_data))
display(model_data[features + [target]].head())

Randuri disponibile: 12705


,temperature,humidity,pressure,pressure_vapor,radiation,precipitation,wind_speed,vane_direction,temperature_depth_5
0,26.5,59.1,98.34,2.046,382.551,0.0,6.4,270,21.74
1,26.8,57.1,98.33,2.009,366.046,0.0,3.2,225,21.76
2,26.9,56.4,98.33,1.992,389.825,0.0,4.0,270,21.76
3,26.8,56.1,98.33,1.975,377.796,0.0,11.2,270,21.78
4,26.7,56.9,98.32,1.992,441.577,0.0,12.0,270,21.77


Am ales variabilele meteorologice pentru a estima temperatura solului ca sa il putem folosi si cand sonda de sol nu furnizeaza o citire.

# Train/Test Split 70/30

In [4]:
split_index = int(len(model_data) * 0.70)
train_data = model_data.iloc[:split_index]
test_data = model_data.iloc[split_index:]
X_train = train_data[features]
X_test = test_data[features]
y_train = train_data[target]
y_test = test_data[target]

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Train period:", train_data["soil_time"].min(), train_data["soil_time"].max())
print("Test period:", test_data["soil_time"].min(), test_data["soil_time"].max())

Train: (8893, 8)
Test: (3812, 8)
Train period: 2025-07-14 12:50:31 2025-11-11 07:09:46
Test period: 2025-11-11 07:14:46 2026-07-14 08:54:56


# Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("Linear Regression")
print(f"MAE: {mean_absolute_error(y_test, lr_pred):.3f} °C")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, lr_pred)):.3f} °C")
print(f"R2: {r2_score(y_test, lr_pred):.3f}")

Linear Regression
MAE: 2.328 °C
RMSE: 3.040 °C
R2: 0.618
